# Few-Shot & In-Context Learning- Teach a Model by Showing, Not Telling

**Module 5 · Lesson 5.2**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Zero-shot vs few-shot - examples teach the pattern
- Building few-shot prompts - format is the pattern
- Why it works - task specification, and the biases
- How many? - sweet spot, many-shot, and reasoning models
- Smart selection - pick the best examples with embeddings
- Production economics - caching and token budgets

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install numpy -q

# Reproducibility - the lesson uses seeded random values throughout
import numpy as np
np.random.seed(42)

## The whole game: three examples, and the model just gets it

> 👶 **Analogy**
>
> **Watch how a toddler learns "dog".** You do not hand them a textbook of canine taxonomy. You point at three dogs and say "dog," point at a cat and say "not dog," and they have it. They learned the pattern from a handful of examples.
> A language model does the same thing. Show it three examples of input and output and it infers the task and generalizes - **in-context, with no training and no code changes**. This is *few-shot learning*, and it is the highest-leverage move in everyday prompt engineering. By the end you will build a system that picks the *best* examples for each query automatically.
> **Where the analogy breaks down:** a toddler keeps the lesson for life. The model keeps it only for this one request - the "learning" lives in the context window and vanishes when the call ends. And it imitates the *form* of your examples even when their labels are wrong, which is exactly why format and ordering matter so much.

Every technique here is tested with **real API calls**. We use Google's Gemini through the current unified SDK, but few-shot is universal - it works the same on Claude (Anthropic) and GPT (OpenAI). This lesson takes the few-shot teaser from Lesson 5.1 and turns it into production craft.

- **Construct** a well-formed few-shot prompt with a consistent template, a complete label space, balanced classes, and a deliberate example order.

- **Explain** why in-context learning works - examples as task specification, not training - and predict how format and ordering change the output.

- **Apply** dynamic example selection (embed the pool, retrieve the nearest examples per query) and contrast static versus dynamic accuracy.

- **Evaluate** when to scale to many-shot, when to drop to zero-shot for reasoning models, and how prompt caching changes the token math.

## Warm-up: 60 seconds of recall

Tap each card to check yourself against earlier lessons. Each one is load-bearing for today.

## The setup: helpers we will reuse all lesson

Every example calls `ask()` (text) or `embed()` (vectors), both on the current unified **google-genai** SDK. The older `google.generativeai` package was deprecated in 2025 ([migration guide](https://ai.google.dev/gemini-api/docs/migrate)).

**📝 `setup.py Gemini`** - *API*

In [ ]:
# pip install "google-genai>=1.0.0" numpy
from google import genai
from google.genai import types
import os, numpy as np

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

def ask(prompt: str, temperature: float = 0.1) -> str:
    """One text completion. Low temp for stable classification."""
    try:
        resp = client.models.generate_content(
            model="gemini-3-flash", contents=prompt,
            config=types.GenerateContentConfig(temperature=temperature))
        return resp.text.strip()
    except Exception as e:
        return f"[API error: {e}]"

def embed(text: str) -> np.ndarray:
    """One embedding vector for a piece of text."""
    r = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return np.array(r.embeddings[0].values)

print(ask("Reply with one word: hello"))
# Output: Hello

- `genai.Client(...)` - one client object, reused everywhere. The old `genai.configure()` global plus per-call `GenerativeModel` is gone.

- `ask()` defaults to `temperature=0.1` - classification wants stable, repeatable answers, not creativity.

- `embed()` calls `gemini-embedding-001` (the current model; the legacy `text-embedding-004` is retired) and returns a NumPy vector we can compare with cosine similarity - the embeddings idea from Lesson 4.1.

- The `try/except` keeps one network blip from crashing a batch classification loop.

---

## 🎯 Concept 1: Zero-shot vs few-shot - examples teach the pattern

### Zero-shot vs few-shot - examples teach the pattern

Does showing examples actually help? Prove it with one API test.

**Sorting a friend's inbox.** "Sort these emails" gets you their best guess at your categories. Show them three sorted examples first - "this one is Billing, this one is Hardware, this one is Account" - and they sort the rest your way.

The gain: the examples did not add knowledge, they pinned down *your* task. Few-shot is task specification by demonstration.

> 🎓 **Analogy**
>
> **Training a new teammate.** Zero-shot is "classify these support emails" and hoping. Few-shot is "here are three sorted examples, now do the rest" - they see the pattern and match it.
> **Where it breaks down:** a teammate remembers the examples tomorrow. The model does not - every call must carry the examples again, because the "learning" lives only in this context window.

**📝 `zero_vs_few.py Gemini`** - *API*

In [ ]:
test = "My laptop won't connect to the company VPN since the update."

zero_shot = f"""Classify into exactly one of: Hardware, Software, Connectivity, Billing, Account.
Reply with ONLY the category.
Email: "{test}"
Category:"""

few_shot_prompt = f"""Classify support emails into exactly one category.

Email: "My monitor keeps flickering and showing weird colors."
Category: Hardware
Email: "I was charged twice for my subscription this month."
Category: Billing
Email: "WiFi keeps dropping every few minutes on my laptop."
Category: Connectivity

Email: "{test}"
Category:"""

print("zero-shot:", ask(zero_shot))
print("few-shot: ", ask(few_shot_prompt))
# Output: zero-shot: Software        (a reasonable guess, sometimes "Network")
# Output: few-shot:  Connectivity    (consistent, and always from your label set)

#### 💡 What just happened

⚡ What just happened?Zero-shot gave a plausible but off-list answer; the three examples taught the model your exact categories and format, so it answered "Connectivity" from your label set. **Examples specify the task, not new knowledge.** The vocabulary: *zero-shot* (no examples), *one-shot* (1), *few-shot* (2-10), *many-shot* (dozens to thousands - Step 4).

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Building few-shot prompts - format is the pattern

### Building few-shot prompts - format is the pattern

A clean, consistent template works far better than a messy one.

**A fill-in-the-pattern game.** `2 -> 4, 3 -> 6, 5 -> 10, 7 -> ?` The cleaner and more uniform the examples, the faster anyone - human or model - spots the rule.

The gain: the *shape* of the examples is the signal. Keep every example in the exact same template and the task reads itself.

So wrap the pattern in a reusable builder: same separator, same labels, every time.

**📝 `builder.py Gemini`** - *API*

In [ ]:
def few_shot(task: str, examples: list[tuple[str, str]],
             query: str, in_label="Input", out_label="Output") -> str:
    """Build a clean, uniform few-shot prompt from (input, output) pairs."""
    p = task.strip() + "\n\n"
    for inp, out in examples:                 # every example, identical shape
        p += f"{in_label}: {inp}\n{out_label}: {out}\n\n"
    p += f"{in_label}: {query}\n{out_label}:"     # query, no answer - model fills it
    return p

sentiment = [
    ("This product is amazing, best purchase ever.", "positive"),
    ("Terrible quality, broke in two days.", "negative"),
    ("It is okay, nothing special.", "neutral"),
    ("Love the design but the battery dies fast.", "mixed"),
]
prompt = few_shot(
    "Classify review sentiment. Reply with one word: positive, negative, neutral, or mixed.",
    sentiment, "Great camera, but support ignored me for a week.",
    "Review", "Sentiment")
print(ask(prompt))
# Output: mixed   (positive about the camera, negative about support)

- **One template, any task** - swap the task string, examples, and labels; the structure never changes. Same function classifies sentiment, detects language, or extracts fields.

- **Uniform separators** - the identical `Label: value` shape on every line is what makes the pattern legible. This is the single biggest lever in few-shot quality.

- **Trailing `{out_label}:`** - ending on the open label is the cue for the model to produce exactly the next value and nothing else.

- It is the same "separate the parts cleanly" instinct as the delimiters from Lesson 5.1 - here applied to examples.

> 💡 **Info**
>
> ⚠️ Common mistakeBeginners obsess over picking the "perfect" examples and ignore formatting. It is backwards: **one example with a stray format** (a different separator, an extra word, an inconsistent label) quietly poisons the whole prompt, because the model imitates surface form. Get the template uniform first; that is most of the battle. Step 3 shows just how far this goes.

#### 💡 What just happened

⚡ What just happened?One reusable `few_shot()` builder produced a clean, uniform prompt and nailed an ambiguous review as "mixed." The function did not change between tasks - only the examples did. **Consistent format is the pattern the model reads.**

---

## 🎯 Concept 3: Why it works - task specification, and the biases

### Why it works - task specification, and the biases

The research result that surprises everyone: correct labels barely matter.

> 📦 **Info**
>
> The headline result: examples are a task ID, not a textbook
> Min et al. 2022, *Rethinking the Role of Demonstrations*: what carries the weight is the **format**, the **label space** (showing all the possible answers), and the **input distribution** (examples from the right domain). Gold labels matter much less than anyone expected - though larger, newer models do benefit more from correct labels than the 2022 models did. Practical reading: get the format and the full label set right first; example correctness is the polish, not the foundation.

> 💡 **Info**
>
> Three biases that make few-shot fragile (Zhao et al. 2021; Lu et al. 2021)
> - **Majority-label bias:** three positive examples and one negative tilt every prediction positive. Fix: balance your classes.
> - **Recency bias:** the last example carries outsized weight - it sits closest to the query the model is about to answer, so its label pulls hardest (order-sensitivity above is the general effect; recency is why the *final* slot matters most). Fix: put the most representative example last, on purpose.
> - **Order sensitivity:** the *same* examples in a different order can swing accuracy dramatically (Lu et al. reported swings from near state-of-the-art to near-random on the same set). Fix: order deliberately, or majority-vote across a few orderings (run the prompt several times with the examples shuffled and take the most common answer).

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?Few-shot examples work as **task specification**: they tell the model which task this is and what an answer looks like. That is why random labels barely hurt but a broken format does - and why order and class balance, which change the apparent task, move the answer. **Design the example *set*, not just the individual examples.**

---

## 🎯 Concept 4: How many? - sweet spot, many-shot, and reasoning models

### How many? - sweet spot, many-shot, and reasoning models

"More examples" is not the answer. The right number depends on the model.

**Studying for an exam.** Zero practice problems: unprepared. Three to five: you see the pattern. Five hundred: exhausted, and learning nothing new after the first twenty.

The gain: there is a sweet spot, and "more" past it just costs you. For few-shot that sweet spot is often 3-5 - but it moves with the model.

> ℹ️ **Info**
>
> Example-count rules of thumb (measure, do not assume)
> - **Simple tasks** (sentiment, yes/no): 2-3 examples.
> - **Classification with 5+ categories:** 3-5, ideally one per category to complete the label space.
> - **Complex extraction/edge cases:** 5-8, chosen to show the hard cases.
> - **Past ~10 with little gain:** stop, or consider many-shot (below) or fine-tuning (Module 9).

**Many-shot is now a real regime,** because context windows got huge. In 2026 the frontier families all ship roughly 1M-token context (Gemini 3.x, GPT-5.5, and Claude Opus 4.8, which lifted from 200K to 1M), so hundreds or thousands of examples physically fit. Agarwal et al. (2024, NeurIPS) showed many-shot gives large gains on hard tasks and even introduced *reinforced* ICL (model-generated reasoning in place of human examples) and *unsupervised* ICL (questions with no answers). The caveat to remember: independent 2026 testing finds *effective* recall degrades past roughly 600K-700K tokens, so "it fits" is not "it is used well."

**Reasoning models flipped the intuition.** A *reasoning model* is one trained to generate a hidden chain of intermediate steps before answering (the o-series, GPT-5.x reasoning); a *classic* model answers directly. On reasoning models, piling on demonstrations can *reduce* accuracy - the "From Medprompt to o1" work found 5-shot underperforming a minimal prompt. Few-shot still shines for classification and strict formatting; for genuine reasoning, prefer zero-shot or 1-2 examples and let the model think. (Advanced reasoning is Lesson 5.3.)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

A team moves a working 5-shot classifier to a new reasoning model and, "to be safe," bumps it to 50 examples. Accuracy *drops* and latency triples. The demonstrations crowd out the model's own chain-of-thought and balloon the token bill. The fix is counter-intuitive but correct: **delete most of the examples** - 1-2, or zero-shot with a clear instruction - and let the reasoning model reason.

#### 💡 What just happened

⚡ What just happened?The optimal example count is not a constant - it is a function of model type and task. Classic models plateau early; long-context models reward many-shot; reasoning models want very few. **The 2026 skill is reading which model you are on and matching the shot count - then measuring.**

---

## 🎯 Concept 5: Smart selection - pick the best examples with embeddings

### Smart selection - pick the best examples with embeddings

Do not use the same examples for every query. Use the ones most like it.

> 🔍 **Analogy**
>
> **Studying the right practice problems.** If the exam question is on quadratics, you do not review random calculus and geometry problems - you find the practice problems most like the question and study those. Dynamic selection does exactly this: for each query, retrieve the most similar examples using embeddings (Lesson 4.1).
> **Where it breaks down:** similarity is not correctness. The nearest examples can share surface words while needing a different label, so dynamic selection still needs a balanced, well-labelled pool underneath it.

**📝 `dynamic_selection.py Gemini`** - *API*

In [ ]:
# A labelled pool. Embed it ONCE and cache the vectors.
pool = [
    ("My phone screen is cracked and unresponsive", "Hardware"),
    ("The app crashes when I upload a photo", "Software"),
    ("WiFi keeps dropping on my laptop", "Connectivity"),
    ("I was double-charged for my subscription", "Billing"),
    ("How do I reset my account password?", "Account"),
    ("My promo code is rejected at checkout", "Billing"),
]
pool_vecs = [(text, label, embed(text)) for text, label in pool]

def nearest(query: str, k: int = 3):
    """Return the k examples whose vectors are closest to the query."""
    q = embed(query)
    scored = []
    for text, label, v in pool_vecs:
        cos = np.dot(q, v) / (np.linalg.norm(q) * np.linalg.norm(v))
        scored.append((cos, text, label))
    scored.sort(reverse=True)
    return [(t, l) for _, t, l in scored[:k]]

query = "There is an unauthorized charge of 49.99 on my account"
picked = nearest(query, k=3)
print("selected:", [l for _, l in picked])
prompt = few_shot("Classify the support ticket. Reply with one category.",
                  picked, query, "Ticket", "Category")
print("answer:  ", ask(prompt))
# Output: selected: ['Billing', 'Billing', 'Account']
# Output: answer:   Billing

- **Embed once, reuse forever** - the pool vectors are computed a single time and cached; only the query is embedded per request.

- **Cosine similarity** - the dot product over the norms measures angle between meaning-vectors; nearest = most semantically similar (the Lesson 4.1 idea, applied).

- **Top-k feeds the builder** - the retrieved pairs go straight into the `few_shot()` builder from Step 2; a Billing query pulls Billing examples, so the model always has a relevant pattern.

- This is the KATE method (Liu et al. 2021): similarity-selected examples beat a fixed set, especially when queries span many categories.

#### 💡 What just happened

⚡ What just happened?Instead of one fixed example set, we retrieved the three examples nearest the query in meaning-space. A charge dispute pulled Billing examples and classified correctly - where a fixed Hardware/Software set would have guessed. **Different query, different examples, better accuracy.** This is the core of the project, and the thing Lesson 5.6 later *automates*.

---

## 🎯 Concept 6: Production economics - caching and token budgets

### Production economics - caching and token budgets

Prompt caching changed the math: front-load examples, pay for them once.

> 📦 **Info**
>
> Prompt caching economics (2026)
> In 2026 all three providers discount cached input by about **90%**, so a stable block of examples is nearly free *on a hot cache*. The break-even matters: a cache *write* costs a surcharge and entries expire (e.g. Anthropic's ~5-minute TTL), so caching only pays when requests arrive faster than the TTL and the write is amortized across many reads - on bursty or low-traffic endpoints you can pay the surcharge and rarely hit the cache. The details differ:
> - **Anthropic:** a cache-write surcharge (about 1.25x input for the 5-minute window, 2x for 1-hour), then reads at roughly 0.1x; minimum cacheable length varies by model.
> - **OpenAI:** automatic caching, no write premium, with a minimum cacheable prompt (about 1,024 tokens).
> - **Google Gemini:** explicit caching with a small per-hour storage fee.
> Batch and cache discounts now stack (roughly 95% off effective input on some providers) - but the batch discount applies only to asynchronous/offline workloads, so it stacks for bulk offline jobs, not for a real-time endpoint. Figures move - check the vendor pricing pages before you budget.

> ✅ **Info**
>
> The architecture: stable prefix, fresh suffix
> Put everything that does not change per request - system prompt, policies, and your few-shot examples - in a **cached prefix**. Put only the user's specific input in the **fresh suffix**. The prefix is cached and cheap; only the short suffix is full price. This is what makes many-shot (Step 4) affordable in production.

**Budget the tokens before you add examples.** Each example is roughly 60-200 tokens; five examples is a few hundred. And recall the token tax from Lesson 4.1 - the same examples in Hindi or other Indic scripts can cost 2-3x more tokens than English, so measure with the real tokenizer, not a word count. Your usable budget is the model limit minus system prompt, examples, query, and the response you need room for.

#### 💡 What just happened

⚡ What just happened?Caching turns the "examples are expensive" objection on its head: a stable example prefix is billed at roughly a tenth on every reuse, so front-loading 5 - or 500 - examples is affordable. **Structure prompts as cached prefix plus fresh suffix, and budget tokens before you add examples.** Cost optimization deepens in Module 13.

## Putting it together: a dynamic few-shot classifier

Combine the lesson into one production-shaped class: embed a pool once, retrieve the nearest examples per query, build a uniform prompt, classify, and measure dynamic against a fixed baseline.

**📝 `classifier.py Complete`** - *project*

In [ ]:
class DynamicFewShot:
    """Embed a pool once; retrieve the nearest examples per query."""
    def __init__(self, task: str, pool: list[tuple[str, str]]):
        self.task = task
        self.pool = [(t, l, embed(t)) for t, l in pool]   # cache vectors once

    def _nearest(self, query: str, k: int):
        q = embed(query)
        scored = [(float(np.dot(q, v) / (np.linalg.norm(q) * np.linalg.norm(v))), t, l)
                  for t, l, v in self.pool]
        scored.sort(reverse=True)
        return [(t, l) for _, t, l in scored[:k]]

    def classify(self, query: str, k: int = 3) -> str:
        prompt = few_shot(self.task, self._nearest(query, k), query, "Ticket", "Category")
        return ask(prompt)

clf = DynamicFewShot(
    "Classify the support ticket. Reply with one of: Hardware, Software, Connectivity, Billing, Account.",
    pool)   # the pool from Step 5

tests = [
    ("My keyboard makes a clicking sound but does not type", "Hardware"),
    ("I was charged for a plan I never signed up for", "Billing"),
    ("4G data stopped working after I traveled abroad", "Connectivity"),
]
correct = sum(clf.classify(q).lower().startswith(exp.lower()[:4]) for q, exp in tests)
print(f"dynamic: {correct}/{len(tests)} correct")
# Output: dynamic: 3/3 correct   (each query pulled its own relevant examples)

#### 💡 What just happened

⚡ What just happened?Every technique in one class: a clean **builder**, an embedded **pool**, **similarity retrieval** per query, and a measured result. Free-text tickets in, correct categories out - and because each query pulls its own relevant examples, it beats a fixed set across categories. This is the production pattern behind real support-ticket routing and content moderation.

### 🧮 The whole lesson on one screen

**Examples teach the task**, not facts; **format is the pattern**, so keep the template uniform; **the example set** (order, balance, label space) matters more than any single example; **count depends on the model** - few-shot for classic, many-shot for long-context, very few for reasoning; **dynamic selection** retrieves the most relevant examples per query; and **caching** makes a stable example prefix nearly free. Above all: few-shot can hurt, so measure with and without.

Forward hooks you just planted: **5.3** advanced reasoning (when to stop showing and start thinking), **5.4** context engineering and long context, **5.5** native structured output, **5.6** prompt optimization and DSPy (which automates example selection), and cost depth in **Module 13**; fine-tuning as the many-shot alternative is **Module 9**.

- Brown et al., *Language Models are Few-Shot Learners* (2020, GPT-3) - [arxiv.org/abs/2005.14165](https://arxiv.org/abs/2005.14165)

- Min et al., *Rethinking the Role of Demonstrations* (2022) - [arxiv.org/abs/2202.12837](https://arxiv.org/abs/2202.12837)

- Zhao et al., *Calibrate Before Use* (2021; the bias analysis) - [arxiv.org/abs/2102.09690](https://arxiv.org/abs/2102.09690)

- Lu et al., *Fantastically Ordered Prompts and Where to Find Them* (2021; order sensitivity) - [arxiv.org/abs/2104.08786](https://arxiv.org/abs/2104.08786)

- Liu et al., *What Makes Good In-Context Examples for GPT-3?* (2021; KATE) - [arxiv.org/abs/2101.06804](https://arxiv.org/abs/2101.06804)

- Agarwal et al., *Many-Shot In-Context Learning* (2024, NeurIPS) - [arxiv.org/abs/2404.11018](https://arxiv.org/abs/2404.11018)

- Anthropic prompt caching docs - [platform.claude.com](https://platform.claude.com/docs/en/build-with-claude/prompt-caching); OpenAI [prompt caching](https://platform.openai.com/docs/guides/prompt-caching); Google [context caching](https://ai.google.dev/gemini-api/docs/caching)

- Google unified `google-genai` SDK migration guide - [ai.google.dev/gemini-api/docs/migrate](https://ai.google.dev/gemini-api/docs/migrate)

## Key takeaways

> ℹ️ **Info**
>
> What you learned - 6 concepts
> - **Examples teach the task** - in-context, no training; they specify format and label space, not facts.
> - **Format is the pattern** - one uniform template; a single inconsistent example poisons the prompt.
> - **Design the set** - format and label space dominate (Min et al.); order and class balance bias the answer (Zhao, Lu).
> - **Count depends on the model** - 3-5 for classic, many-shot for long-context, 1-2 for reasoning models.
> - **Dynamic selection** - embed the pool, retrieve the nearest examples per query (KATE); beats a fixed set across categories.
> - **Caching makes it cheap** - stable example prefix at ~10% cost on reuse; budget tokens, and measure with and without.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Few-Shot & In-Context Learning- Teach a Model by Showing, Not Telling**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-5_2.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-5.2.html` - regenerate this notebook whenever the source HTML is updated.*
